In [0]:

# ===================================================
# BLOCK 1 — IMPORTS AND PARAMETERS (PYTHON)
# ===================================================

"""
Capture Lakeflow Job execution identifiers used to correlate operational logs,
quality metrics, retries, and repair runs.
"""

from datetime import datetime, timezone
from uuid import uuid4

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("job_id", "MANUAL", "Lakeflow Job ID")
dbutils.widgets.text("job_run_id", "MANUAL", "Lakeflow Job Run ID")
dbutils.widgets.text("environment", "development", "Environment")
dbutils.widgets.text("run_mode", "validation", "Run mode")

JOB_ID = dbutils.widgets.get("job_id")
JOB_RUN_ID = dbutils.widgets.get("job_run_id")
ENVIRONMENT = dbutils.widgets.get("environment")
RUN_MODE = dbutils.widgets.get("run_mode")

# Generate a unique identifier during interactive development. Lakeflow Jobs
# replaces this value with its native job run ID during orchestrated execution.
if JOB_RUN_ID == "MANUAL" or JOB_RUN_ID.startswith("{{"):
    JOB_RUN_ID = f"manual-{uuid4()}"

STARTED_AT_UTC = datetime.now(timezone.utc)

print(f"Job ID: {JOB_ID}")
print(f"Job run ID: {JOB_RUN_ID}")
print(f"Environment: {ENVIRONMENT}")
print(f"Run mode: {RUN_MODE}")

In [0]:

# ===================================================
# BLOCK 2 — OPERATIONAL TABLES (PYTHON)
# ===================================================

"""
Create durable run-level and metric-level audit tables independently from the
pipeline schemas so operational evidence survives pipeline refreshes.
"""

spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS semiconplus_portfolio.operations
    COMMENT 'Workflow execution logs, validation metrics, and operational controls'
    """
)

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS
    semiconplus_portfolio.operations.workflow_run_log
    (
        job_run_id STRING NOT NULL,
        job_id STRING,
        environment STRING,
        run_mode STRING,
        run_status STRING NOT NULL,
        started_at_utc TIMESTAMP NOT NULL,
        completed_at_utc TIMESTAMP,
        initiated_by STRING,
        bronze_row_count BIGINT,
        silver_row_count BIGINT,
        late_row_count BIGINT,
        quarantine_row_count BIGINT,
        gold_row_count BIGINT,
        failure_message STRING,
        last_updated_at_utc TIMESTAMP NOT NULL
    )
    USING DELTA
    COMMENT 'One auditable status record for each SemiconPlus workflow run'
    """
)

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS
    semiconplus_portfolio.operations.workflow_quality_metrics
    (
        job_run_id STRING NOT NULL,
        metric_name STRING NOT NULL,
        metric_value DOUBLE,
        expected_condition STRING,
        validation_status STRING NOT NULL,
        recorded_at_utc TIMESTAMP NOT NULL
    )
    USING DELTA
    COMMENT 'Data-quality and reconciliation metrics produced by workflow runs'
    """
)

In [0]:
# ===================================================
# BLOCK 3 — IDEMPOTENT RUN REGISTRATION (PYTHON)
# ===================================================

"""
Register the workflow run with explicit target-column mappings so nullable
completion fields and data-volume metrics remain empty until finalization.
MERGE semantics prevent task retries from creating duplicate run records.
"""

run_schema = T.StructType(
    [
        T.StructField("job_run_id", T.StringType(), False),
        T.StructField("job_id", T.StringType(), True),
        T.StructField("environment", T.StringType(), True),
        T.StructField("run_mode", T.StringType(), True),
        T.StructField("run_status", T.StringType(), False),
        T.StructField("started_at_utc", T.TimestampType(), False),
        T.StructField("initiated_by", T.StringType(), True),
        T.StructField("last_updated_at_utc", T.TimestampType(), False),
    ]
)

run_source = spark.createDataFrame(
    [
        (
            JOB_RUN_ID,
            JOB_ID,
            ENVIRONMENT,
            RUN_MODE,
            "RUNNING",
            STARTED_AT_UTC,
            spark.sql("SELECT session_user()").first()[0],
            STARTED_AT_UTC,
        )
    ],
    run_schema,
)

run_log = DeltaTable.forName(
    spark,
    "semiconplus_portfolio.operations.workflow_run_log",
)

(
    run_log.alias("target")
    .merge(
        run_source.alias("source"),
        "target.job_run_id = source.job_run_id",
    )
    .whenMatchedUpdate(
        set={
            "job_id": "source.job_id",
            "environment": "source.environment",
            "run_mode": "source.run_mode",
            "run_status": "'RUNNING'",
            "completed_at_utc": "CAST(NULL AS TIMESTAMP)",
            "failure_message": "CAST(NULL AS STRING)",
            "last_updated_at_utc": "source.last_updated_at_utc",
        }
    )
    .whenNotMatchedInsert(
        values={
            "job_run_id": "source.job_run_id",
            "job_id": "source.job_id",
            "environment": "source.environment",
            "run_mode": "source.run_mode",
            "run_status": "source.run_status",
            "started_at_utc": "source.started_at_utc",
            "completed_at_utc": "CAST(NULL AS TIMESTAMP)",
            "initiated_by": "source.initiated_by",
            "bronze_row_count": "CAST(NULL AS BIGINT)",
            "silver_row_count": "CAST(NULL AS BIGINT)",
            "late_row_count": "CAST(NULL AS BIGINT)",
            "quarantine_row_count": "CAST(NULL AS BIGINT)",
            "gold_row_count": "CAST(NULL AS BIGINT)",
            "failure_message": "CAST(NULL AS STRING)",
            "last_updated_at_utc": "source.last_updated_at_utc",
        }
    )
    .execute()
)

print("Workflow run registered.")

In [0]:

# ===================================================
# BLOCK 4 — PUBLISH TASK VALUES (PYTHON)
# ===================================================

"""
Expose the normalized run identifier to downstream tasks and to the Lakeflow
Job run details for operational traceability.
"""

try:
    dbutils.jobs.taskValues.set(
        key="normalized_job_run_id",
        value=JOB_RUN_ID,
    )
    dbutils.jobs.taskValues.set(
        key="run_started_at_utc",
        value=STARTED_AT_UTC.isoformat(),
    )
except Exception:
    print("Task values are unavailable during interactive execution.")

print("INITIALIZE RUN: PASSED")